In [1]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  17.6M      0 --:--:-- --:--:-- --:--:-- 17.7M


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.0 MB/s eta 0:00:00


In [4]:
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics

from IPython.display import Video, display
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
VIDEO_PATH = Path("/content/development.mp4")

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_PATH = OUTPUT_DIR / "bytetrack_baseline.mp4"

TRACKS_CSV_PATH = OUTPUT_DIR / "tracks.csv"

MODEL_NAME = "yolo26n.pt"
TRACKER_CONFIG = "bytetrack.yaml"

IMAGE_SIZE = 640
DETECTION_CONFIDENCE = 0.10
NMS_IOU_THRESHOLD  = 0.70

TARGET_CLASS_NAMES = {
    "person",
    "bicycle",
    "car",
}

DEVICE = torch.device(0 if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

Device: cuda:0
GPU: Tesla T4


**Model and vehicle class IDs**

In [ ]:
class_lookup_model = YOLO(MODEL_NAME)

model_names = class_lookup_model.names

if isinstance(model_names, dict):
    class_id_to_name = {
        int(class_id): str(class_name)
        for class_id, class_name in model_names.items()
    }
else:
    class_id_to_name = {
        class_id: str(class_name)
        for class_id, class_name in enumerate(model_names)
    }

TARGET_CLASS_IDS = sorted(
    class_id 
    for class_id, class_name in class_id_to_name.items()
    if class_name.lower() in TARGET_CLASS_NAMES
)

selected_classes_df = pd.DataFrame([
    {
        "class_id": class_id,
        "class_name": class_id_to_name[class_id]
    }
    for class_id in TARGET_CLASS_IDS
])

display(selected_classes_df)
print("Selected class IDs:", TARGET_CLASS_IDS)

,class_id,class_name
0,0,person
1,1,bicycle
2,2,car


Selected class IDs: [0, 1, 2]


## Main tracking pipeline

**YOLO + ByteTrack**

In [9]:
tracking_model = YOLO(MODEL_NAME)
capture = cv2.VideoCapture(str(VIDEO_PATH))

width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

source_fps = float(capture.get(cv2.CAP_PROP_FPS))
reported_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO_PATH),
    fourcc, source_fps, (width, height)
)

if not writer.isOpened():
    capture.release()
    raise RuntimeError("Could not open VideoWriter")

In [11]:
track_columns = [
    "frame_index",
    "timestamp_seconds",
    "track_id",
    "class_id",
    "class_name",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",
    "center_x",
    "center_y",
]

track_rows = []
processed_frames = 0
frames_with_track_ids = 0
frames_without_track_ids = 0

frame_processing_times_ms = []
total_start = perf_counter()

try:
    while True:
        success, frame = capture.read()
        if not success:
            break
        
        frame_start = perf_counter()
        frame_index = processed_frames
        
        timestamp_seconds = frame_index / source_fps
        
        result = tracking_model.track(
            source=frame, 
            persist=True,
            tracker=TRACKER_CONFIG,
            classes=TARGET_CLASS_IDS,
            conf=DETECTION_CONFIDENCE,
            iou=NMS_IOU_THRESHOLD,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False
        )[0]
        
        annotated_frame = result.plot()
        
        boxes = result.boxes
        
        has_track_ids = (
            boxes is not None
            and len(boxes) > 0
            and boxes.id is not None
        )
        
        if has_track_ids:
            frames_with_track_ids += 1

            xyxy_values = boxes.xyxy.detach().cpu().numpy()
            
            confidence_values = boxes.conf.detach().cpu().numpy()
            
            class_id_values = boxes.cls.detach().cpu().numpy().astype(int)
            
            track_id_values = boxes.id.detach().cpu().numpy().astype(int)
            
            for (xyxy, confidence, class_id, track_id) in zip(
                xyxy_values, confidence_values, class_id_values, track_id_values
            ):
                x1, y1, x2, y2 = map(float, xyxy)
                
                center_x = (x1 + x2) / 2.0
                center_y = (y1 + y2) / 2.0
                
                track_rows.append({
                    "frame_index": frame_index,
                    "timestamp_seconds": timestamp_seconds,
                    "track_id":  int(track_id),
                    "class_id": int(class_id),
                    "class_name": (
                        class_id_to_name[ int(class_id)]
                    ),
                    "confidence": float(confidence),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "center_x": center_x,
                    "center_y": center_y,
                })
        
        else:
            frames_without_track_ids += 1
        
        cv2.putText(
            annotated_frame, 
            (
                f"Frame: {frame_index}  "
                f"Time: "
                f"{timestamp_seconds:.2f}s"
            ),
            (25, 45), cv2.FONT_HERSHEY_SIMPLEX,
            0.9, (0, 255, 0), 2, cv2.LINE_AA
        )
        
        writer.write(annotated_frame)
        frame_end = perf_counter()
        
        frame_processing_times_ms.append(
            (frame_end - frame_start) * 1000
        )
        
        processed_frames += 1
        
        if processed_frames % 100 == 0:
            print(
                f"Processed "
                f"{processed_frames}/"
                f"{reported_frames} frames"
            )

finally:
    capture.release()
    writer.release()

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 250ms
Prepared 1 package in 52ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 100/647 frames
Processed 200/647 frames
Processed 300/647 frames
Processed 400/647 frames
Processed 500/647 frames
Processed 600/647 frames


In [12]:
total_processing_seconds = perf_counter() - total_start

tracks_df = pd.DataFrame(
    track_rows,
    columns=track_columns   
)

tracks_df.to_csv(TRACKS_CSV_PATH, index=False)

processing_fps = processed_frames / total_processing_seconds
real_time_factor = processing_fps / source_fps

print("Processed frames:", processed_frames)
print("Track observations:", len(tracks_df))
print("Output video:", OUTPUT_VIDEO_PATH)
print("Tracks CSV:", TRACKS_CSV_PATH)

Processed frames: 647
Track observations: 306
Output video: /content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02/bytetrack_baseline.mp4
Tracks CSV: /content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02/tracks.csv


**Overall summary**

In [13]:
unique_track_ids = (
    tracks_df["track_id"].nunique()
    if not tracks_df.empty else 0
)

summary_df = pd.DataFrame(
    [
        {
            "reported_input_frames": reported_frames,
            "processed_frames": processed_frames,
            "frames_with_track_ids": frames_with_track_ids,
            "frames_without_track_ids": frames_without_track_ids,
            "track_observations": len(tracks_df),
            "unique_track_ids": unique_track_ids,
            "mean_latency_ms": np.mean(frame_processing_times_ms),
            "p95_latency_ms": np.percentile(
                frame_processing_times_ms, 95
            ),
            "processing_fps": processing_fps,
            "source_fps": source_fps,
            "real_time_factor": real_time_factor
        }
    ]
)

display(summary_df.round(3))

,reported_input_frames,processed_frames,frames_with_track_ids,frames_without_track_ids,track_observations,unique_track_ids,mean_latency_ms,p95_latency_ms,processing_fps,source_fps,real_time_factor
0,647,647,231,416,306,10,29.101,24.448,6.92,12.0,0.577


**Track-level summary**

In [ ]:
def dominant_value(series):
    modes = series.mode()

    if len(modes) > 0:
        return modes.iloc[0]

    return series.iloc[0]


track_summary_df = (
    tracks_df.groupby("track_id", as_index=False).agg(
        dominant_class=("class_name", dominant_value),
        first_frame=("frame_index","min"),
        last_frame=("frame_index", "max"),
        observations=( "frame_index","size"),
        mean_confidence=("confidence", "mean"),
    )
)

track_summary_df["span_frames"] = (
    track_summary_df["last_frame"]
    - track_summary_df["first_frame"]
    + 1
)

track_summary_df["coverage_ratio"] = (
    track_summary_df["observations"] / track_summary_df["span_frames"]
)

top_five_tracks = (
    track_summary_df.sort_values("observations", ascending=False).head(5).reset_index(drop=True)
)

display(top_five_tracks.round(3))

,track_id,dominant_class,first_frame,last_frame,observations,mean_confidence,span_frames,coverage_ratio
0,35,person,485,555,71,0.706,71,1.000
1,1,person,16,84,69,0.824,69,1.000
2,36,car,528,589,57,0.775,62,0.919
3,11,car,203,229,27,0.741,27,1.000
4,50,person,570,592,19,0.418,23,0.826
